# 4. ICA as a spatial filter: visual and motor components

**Idea.** ICA splits each participant's EEG into components. Each component is a fixed weighting of the electrodes, i.e. a **spatial filter**: its time course is one clean "virtual channel". So far ICA was used to *remove* artifact components. Here it is used to *keep* the brain components we care about (the game has no sound, so no auditory components):

- a **visual component** (alpha, 8–12 Hz);
- a **motor component** (mu, 8–12 Hz, and/or beta, 13–30 Hz).

**Which ICA.** A component can come from the session ICA (`01_ica`) or from a localizer ICA (`04a_localizer_ica`: `motor` = bubble popping, `visual` = instruction video). The choice says which.

**No automatic scoring.** The notebook saves one overview PNG with the scalp maps of **all kept components** (the property plot of every component, with time course and spectrum, is already saved by `01_ica` in `<sub>/components/`). The choice is made by eye, with the supervisor, and written into `SPATIAL_FILTER_CHOICE` in `config.py`. Only then are the envelopes computed.

**Steps:** 1. load clean data + ICA → 2. one overview PNG with the scalp maps of all kept components → 3. envelopes of the chosen components (after the review), downsampled to `ENVELOPE_SFREQ` (25 Hz) for cRQA → 4. save.

Bands are in `config.py` (`SPATIAL_FILTER_BANDS`).

In [ ]:
import importlib
import os

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config as cfg
%matplotlib inline

## 0. Participant

In [ ]:
# execute_notebooks.py sets RT_SUBJECT
SUBJECT = os.environ.get("RT_SUBJECT", "sub-015")
SESSION = cfg.session_of(SUBJECT)
OUT_DIR = cfg.out_dir(SUBJECT)
REVIEW_DIR = cfg.OUT_ROOT / "spatial_filter_review" / SESSION  # one folder per session for the review
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print(SUBJECT, f"({SESSION})", "->", REVIEW_DIR)

## 1. Load the clean data and the ICA

In [ ]:
raw = mne.io.read_raw(OUT_DIR / f"{SUBJECT}_clean_raw.fif", preload=True)
ica = mne.preprocessing.read_ica(OUT_DIR / f"{SUBJECT}_ica.fif")
removed = pd.read_csv(OUT_DIR / f"{SUBJECT}_badcomps.csv")["component"].astype(int).tolist()
kept = [c for c in range(ica.n_components_) if c not in removed]

sources = ica.get_sources(raw)
print(f"{len(kept)} kept components; removed as artifacts: {removed}")

## 2. Overview: the scalp maps of all kept components in one PNG

Only the maps (no spectra), all kept components of this participant side by side, so they are easy to compare. Saved as `spatial_filter_review/<session>/<subject>_all_maps.png`.

In [ ]:
figure = ica.plot_components(picks=kept, ncols=6, show=False)  # with picks given, MNE puts all maps in one figure
figure = figure[0] if isinstance(figure, list) else figure
figure.suptitle(f"{SUBJECT}: scalp maps of the kept components", fontsize=11)
figure.savefig(REVIEW_DIR / f"{SUBJECT}_all_maps.png", dpi=100, bbox_inches="tight")
plt.close(figure)

## 3. Amplitude envelopes of the chosen components (after the review)

Write the choice into `SPATIAL_FILTER_CHOICE` in `config.py` as `{area: (ICA, component, band)}`, where ICA is `"session"` (the ICA of `01_ica`) or `"motor"` / `"visual"` (the localizer ICAs of `04a`), e.g. `"sub-013": {"visual": ("session", 5, "alpha"), "motor": ("motor", 3, "beta")}`, then run from here. Without a choice this step and the next do nothing.

For each chosen component:
1. its time course on the **clean data** (the ICA's unmixing applied to `clean_raw`);
2. band-pass (`SPATIAL_FILTER_BANDS`) with 1 Hz slopes (`SPATIAL_FILTER_TRANSITION`; MNE's default slopes are wider, so mu/alpha around 11–12 Hz leaked into beta), then the Hilbert envelope, on the **continuous** signal (no edge artifacts per trial);
3. **downsample** to `ENVELOPE_SFREQ` (25 Hz). cRQA compares every time point with every other one: a 300 s trial has 150 000 points at 500 Hz (a recurrence matrix of 2 × 10¹⁰ cells, too large), 7 500 at 25 Hz. The envelope changes slowly, so little is lost; resampling low-passes at 12.5 Hz first (no aliasing).

Components have an arbitrary sign and scale; the envelope removes the sign, and cRQA will use standardized envelopes. Bad stretches (`BAD_amplitude`) are still in the envelope as annotations; they are handled when the trials are cut.

In [ ]:
importlib.reload(cfg)  # pick up changes made to config.py
CHOSEN = cfg.SPATIAL_FILTER_CHOICE.get(SUBJECT, {})
if not CHOSEN:
    print(f"{SUBJECT}: no choice in SPATIAL_FILTER_CHOICE yet; envelopes skipped")


def ica_file(name):
    """The ICA a component comes from: "session" (01_ica) or a localizer ("motor", "visual"; 04a_localizer_ica)."""
    if name == "session":
        return OUT_DIR / f"{SUBJECT}_ica.fif"
    assert name in cfg.LOCALIZERS, f"unknown ICA {name!r}: use 'session' or one of {list(cfg.LOCALIZERS)}"
    return OUT_DIR / f"{SUBJECT}_{name}_localizer_ica.fif"


envelopes = {}
for area, (ica_name, comp, band) in CHOSEN.items():
    if ica_name == "session":
        assert comp not in removed, f"{area}: session component {comp} was removed as an artifact"
    source_ica = mne.preprocessing.read_ica(ica_file(ica_name))
    x = source_ica.get_sources(raw).pick([f"ICA{comp:03d}"])     # the component's time course on the clean data
    # band-pass with narrow slopes (SPATIAL_FILTER_TRANSITION, 1 Hz), so neighbouring bands do not leak in;
    # components are 'misc' channels: picks needed
    x.filter(*cfg.SPATIAL_FILTER_BANDS[area][band], picks="all",
             l_trans_bandwidth=cfg.SPATIAL_FILTER_TRANSITION, h_trans_bandwidth=cfg.SPATIAL_FILTER_TRANSITION)
    x.apply_hilbert(envelope=True, picks="all")
    # MNE 1.13: the Raw from get_sources cannot be resampled (it keeps its first sample as a list), so the
    # envelope is copied into a new Raw with the same data, first sample and events
    events_kept = x.annotations
    x = mne.io.RawArray(x.get_data(), x.info, first_samp=x.first_samp, verbose="error")
    x.set_annotations(events_kept)
    x.resample(cfg.ENVELOPE_SFREQ)                                # 500 Hz -> 25 Hz; events are kept
    envelopes[area] = x
    print(f"{area}: {ica_name} ICA{comp:03d}, {band} -> {x.info['sfreq']:.0f} Hz, {x.n_times} samples")

if envelopes:
    fig, ax = plt.subplots(figsize=(10, 3))
    t0 = 60
    for area, x in envelopes.items():
        sf = x.info["sfreq"]
        seg = x.get_data(start=int(t0 * sf), stop=int((t0 + 10) * sf))[0]
        ica_name, comp, band = CHOSEN[area]
        ax.plot(np.linspace(t0, t0 + 10, seg.size), seg / seg.std(), label=f"{area}: {ica_name} ICA{comp:03d}, {band}")
    ax.set(xlabel="time (s)", ylabel="envelope (z)"); ax.legend()

### 3b. From component to envelope, step by step (for understanding; nothing is saved)

The same steps as above, kept apart and plotted for a few seconds of one area:

1. **Component**: the component's time course on the clean data (500 Hz), all frequencies.
2. **Band-passed**: only the band of the envelope (e.g. alpha 8-12 Hz), same filter as above.
3. **Envelope**: the Hilbert envelope traced over the band-passed signal (500 Hz), and the saved envelope after resampling to 25 Hz (dots).
4. **Smoothed envelope** (for viewing only, not saved or used by 05/06): the envelope low-passed at `SMOOTH_HZ` (1 Hz), over the saved envelope. It keeps the slow ups and downs over seconds and removes the fast bumps (~0.3-1 s), which arise in any narrow band (an 8-12 Hz band 'beats' at about its width, 4 Hz) and are unlikely to be shared between two people.

Change `PLOT_AREA`, `SMOOTH_HZ`, `T0` (start, s from the start of the clean data) and `DURATION` (s) to look elsewhere. A short window (a few s) shows the single oscillations; a longer one (30-60 s) shows the slow ups and downs that cRQA compares.

In [ ]:
PLOT_AREA = "visual"   # an area of SPATIAL_FILTER_CHOICE: "visual" or "motor"
T0 = 450               # start [s], from the start of the clean data
DURATION = 30          # length [s]
SMOOTH_HZ = 1          # panel 4: low-pass of the envelope [Hz] (viewing only)

if PLOT_AREA in CHOSEN:
    ica_name, comp, band = CHOSEN[PLOT_AREA]
    source_ica = mne.preprocessing.read_ica(ica_file(ica_name))
    # the three steps of section 3, each on its own copy (filtered on the whole recording, as above, so the window
    # has no filter edge effects; only the plot is cut to the window)
    step1 = source_ica.get_sources(raw).pick([f"ICA{comp:03d}"])                    # 1. component, all frequencies
    step2 = step1.copy().filter(*cfg.SPATIAL_FILTER_BANDS[PLOT_AREA][band], picks="all",   # 2. band-passed
                                l_trans_bandwidth=cfg.SPATIAL_FILTER_TRANSITION,
                                h_trans_bandwidth=cfg.SPATIAL_FILTER_TRANSITION, verbose="error")
    step3 = step2.copy().apply_hilbert(envelope=True, picks="all")                 # 3. envelope, 500 Hz
    saved = envelopes[PLOT_AREA]                                                   # the saved envelope, 25 Hz
    # 4. smoothed envelope: low-pass of the 500 Hz envelope. Narrow slope (half of SMOOTH_HZ), so that at 1 Hz the
    # -6 dB point is 1.25 Hz; MNE's default slope (2 Hz) would let bumps up to ~2 Hz through
    step4 = step3.copy().filter(None, SMOOTH_HZ, picks="all", h_trans_bandwidth=SMOOTH_HZ / 2, verbose="error")


    def window(x):
        """Times [s] and values of the window T0..T0+DURATION of a one-channel Raw (any sampling rate)."""
        sf = x.info["sfreq"]
        start, stop = int(T0 * sf), int((T0 + DURATION) * sf)
        return x.times[start:stop], x.get_data(start=start, stop=stop)[0]


    fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
    t, y = window(step1)
    axes[0].plot(t, y, color="0.3", lw=0.8)
    axes[0].set(title=f"1. {ica_name} ICA{comp:03d}: component time course (all frequencies, 500 Hz)")
    t, y = window(step2)
    lo, hi = cfg.SPATIAL_FILTER_BANDS[PLOT_AREA][band]
    axes[1].plot(t, y, color="C0", lw=0.8)
    axes[1].set(title=f"2. band-passed {lo}-{hi} Hz ({band})")
    axes[2].plot(t, y, color="C0", lw=0.6, alpha=0.5, label="band-passed")
    t, y = window(step3)
    axes[2].plot(t, y, color="C3", lw=1.5, label="Hilbert envelope (500 Hz)")
    t, y = window(saved)
    axes[2].plot(t, y, "o", color="k", ms=3, label=f"saved envelope ({saved.info['sfreq']:.0f} Hz)")
    axes[2].set(title="3. envelope: the outline of the band-passed signal")
    axes[2].legend(loc="upper right", fontsize=8)
    axes[3].plot(t, y, color="k", lw=0.6, alpha=0.4, label=f"saved envelope ({saved.info['sfreq']:.0f} Hz)")
    t, y = window(step4)
    axes[3].plot(t, y, color="C2", lw=2, label=f"low-passed at {SMOOTH_HZ} Hz")
    axes[3].set(title=f"4. smoothed envelope: only the changes slower than {SMOOTH_HZ} Hz (viewing only, not saved)",
                xlabel="time (s)")
    axes[3].legend(loc="upper right", fontsize=8)
    for ax in axes:
        ax.set_ylabel("a.u.")   # ICA components have no physical unit
    fig.tight_layout()
else:
    print(f"{PLOT_AREA} not in SPATIAL_FILTER_CHOICE for {SUBJECT}: choose one of {list(CHOSEN)}")

### 3c. Why smooth the envelope? The envelope without smoothing, zoomed in, next to a noise control (for understanding; nothing is saved)

The question: the envelope already tells how strong the rhythm is at each moment, so why low-pass it again before cRQA?

**What the plots show** (same component and area as 3b):

1. **60 s of the envelope as saved (25 Hz, no smoothing)**, with the smoothed one (1 Hz) on top. The unsmoothed envelope is full of fast bumps, several per second.
2. **Zoom (4 s)**: the band-passed signal and its envelope. Each bump is a short burst of the rhythm: the oscillation grows and shrinks a few times per second.
3. **The same zoom for a noise control**: the same component with its phases shuffled. It has exactly the same spectrum (so the same amount of alpha), but any real waxing and waning of the rhythm is destroyed: it is noise that only *looks* like alpha. Its band-passed envelope has **the same fast bumps**.
4. **The noise control's envelope over the same 60 s**: it looks just like panel 1.
5. **How fast the envelope changes** (spectrum of the envelope): the component vs the noise control, with the smoothing cut-off. Above ~1 Hz the two curves lie on top of each other; only below ~1 Hz does the real component have more power.

**Why this argues for smoothing.** Any signal squeezed through a narrow band (8-12 Hz is 4 Hz wide) gets an envelope that bumps up and down at up to about the band's width (here up to ~4 Hz): neighbouring frequencies inside the band drift in and out of step ("beating"). This happens even for pure noise, so the fast bumps mostly reflect the filter, not the brain, and they are random from moment to moment: two people's fast bumps cannot match except by chance, which only adds noise to cRQA. What distinguishes the real component from the noise control is the **slow** part (below ~1 Hz): the rhythm being stronger or weaker for seconds at a time (attention, arousal, task). That is the part where two people could be coupled, and it is what the 1 Hz low-pass keeps.

**What smoothing does not do.** Even the noise control has slow ups and downs (in tests 36-38 % of its envelope variance lies below 1 Hz), so smoothing does not remove chance similarity between two people; it removes the part that is *only* chance. Chance coupling of the slow part is what the surrogate pairs in cRQA control for.

The printed table gives, for both, the share of the envelope's variance below and above the cut-off, and how much more slow power the real component has than the noise. Change `ZOOM_T0`, `ZOOM` and `LONG` to look elsewhere (3b must be run first: this cell uses its `step1`-`step4`, `saved`, `T0`, `SMOOTH_HZ`).

In [ ]:
LONG = 60              # length of the overview [s], from T0 (3b)
ZOOM_T0 = T0 + 20      # start of the zoom [s]
ZOOM = 4               # length of the zoom [s]

if PLOT_AREA in CHOSEN:
    lo, hi = cfg.SPATIAL_FILTER_BANDS[PLOT_AREA][band]

    # Noise control: the component with the same spectrum but random phases (fixed seed). Same amount of power at
    # every frequency, but none of the real waxing and waning of the rhythm.
    rng = np.random.default_rng(cfg.RANDOM_STATE)
    x = step1.get_data()[0]
    spectrum = np.fft.rfft(x)
    phases = np.exp(2j * np.pi * rng.random(spectrum.size))
    phases[0] = 1                                    # keep the mean
    noise = mne.io.RawArray(np.fft.irfft(spectrum * phases, n=x.size)[np.newaxis], step1.info, verbose="error")
    noise_band = noise.copy().filter(lo, hi, picks="all", l_trans_bandwidth=cfg.SPATIAL_FILTER_TRANSITION,
                                     h_trans_bandwidth=cfg.SPATIAL_FILTER_TRANSITION, verbose="error")
    noise_env = noise_band.copy().apply_hilbert(envelope=True, picks="all")
    noise_smooth = noise_env.copy().filter(None, SMOOTH_HZ, picks="all", h_trans_bandwidth=SMOOTH_HZ / 2,
                                           verbose="error")

    def cut(x, start, length):
        """Times [s] and values of start..start+length of a one-channel Raw (any sampling rate)."""
        sf = x.info["sfreq"]
        a, b = int(start * sf), int((start + length) * sf)
        return x.times[a:b], x.get_data(start=a, stop=b)[0]

    fig = plt.figure(figsize=(12, 13))
    grid = fig.add_gridspec(4, 2)
    ax_long = fig.add_subplot(grid[0, :])
    ax_zoom = fig.add_subplot(grid[1, 0])
    ax_zoom_noise = fig.add_subplot(grid[1, 1], sharey=ax_zoom)
    ax_long_noise = fig.add_subplot(grid[2, :], sharey=ax_long)
    ax_spec = fig.add_subplot(grid[3, :])

    # 1. 60 s of the saved envelope (no smoothing) and the smoothed one
    t, y = cut(saved, T0, LONG)
    ax_long.plot(t, y, color="C3", lw=0.8, label=f"envelope as saved ({saved.info['sfreq']:.0f} Hz, no smoothing)")
    t, y = cut(step4, T0, LONG)
    ax_long.plot(t, y, color="C2", lw=2.2, label=f"smoothed (low-pass {SMOOTH_HZ} Hz)")
    ax_long.axvspan(ZOOM_T0, ZOOM_T0 + ZOOM, color="0.85", zorder=0, label="zoom below")
    ax_long.set(title=f"1. {SUBJECT}, {PLOT_AREA} ({ica_name} ICA{comp:03d}, {lo}-{hi} Hz): envelope, {LONG} s",
                ylabel="a.u.")
    ax_long.legend(loc="upper right", fontsize=8)

    # 2. zoom: band-passed signal, its envelope (500 Hz), the saved samples, the smoothed envelope
    def zoom(ax, band_passed, envelope, smoothed, title, dots=None):
        t, y = cut(band_passed, ZOOM_T0, ZOOM)
        ax.plot(t, y, color="C0", lw=0.7, alpha=0.7, label=f"band-passed {lo}-{hi} Hz")
        t, y = cut(envelope, ZOOM_T0, ZOOM)
        ax.plot(t, y, color="C3", lw=1.6, label="envelope (no smoothing)")
        if dots is not None:
            t, y = cut(dots, ZOOM_T0, ZOOM)
            ax.plot(t, y, "o", color="k", ms=3, label=f"saved samples ({dots.info['sfreq']:.0f} Hz)")
        t, y = cut(smoothed, ZOOM_T0, ZOOM)
        ax.plot(t, y, color="C2", lw=2.2, label=f"smoothed ({SMOOTH_HZ} Hz)")
        ax.set(title=title, xlabel="time (s)", ylabel="a.u.")
        ax.legend(loc="upper right", fontsize=7)

    zoom(ax_zoom, step2, step3, step4, f"2. zoom ({ZOOM} s): real component", dots=saved)
    zoom(ax_zoom_noise, noise_band, noise_env, noise_smooth, f"3. zoom ({ZOOM} s): noise control (phases shuffled)")

    # 3. the noise control's envelope over the same 60 s
    t, y = cut(noise_env, T0, LONG)
    ax_long_noise.plot(t, y, color="C3", lw=0.8, label="envelope (no smoothing)")
    t, y = cut(noise_smooth, T0, LONG)
    ax_long_noise.plot(t, y, color="C2", lw=2.2, label=f"smoothed ({SMOOTH_HZ} Hz)")
    ax_long_noise.set(title=f"4. noise control: same spectrum, phases shuffled, {LONG} s: the same fast bumps",
                      xlabel="time (s)", ylabel="a.u.")
    ax_long_noise.legend(loc="upper right", fontsize=8)

    # 4. spectrum of the envelope: how fast it changes (whole recording; bad stretches left out for the real one)
    FMAX = min(25, max(6, 1.5 * (hi - lo)))   # up to past the band width (alpha 6 Hz, beta 25 Hz)
    def envelope_spectrum(env, reject):
        psd = env.compute_psd(method="welch", fmin=0.05, fmax=FMAX, n_fft=int(20 * env.info["sfreq"]), picks="all",
                              reject_by_annotation=reject, verbose="error")
        return psd.freqs, psd.get_data()[0]

    f, p_real = envelope_spectrum(step3, True)
    _, p_noise = envelope_spectrum(noise_env, False)
    ax_spec.semilogy(f, p_real, color="C3", lw=1.5, label="real component")
    ax_spec.semilogy(f, p_noise, color="0.4", lw=1.5, ls="--", label="noise control (same spectrum, phases shuffled)")
    ax_spec.axvspan(0, SMOOTH_HZ, color="C2", alpha=0.12, label=f"kept by the smoothing (below {SMOOTH_HZ} Hz)")
    ax_spec.axvline(hi - lo, color="0.6", lw=1, ls=":", label=f"band width ({hi - lo} Hz)")
    ax_spec.set(title="5. how fast the envelope changes: spectrum of the envelope (above 1 Hz: real = noise)", xlabel="frequency of the ups and "
                "downs of the envelope (Hz)", ylabel="power (a.u.)")
    ax_spec.legend(loc="upper right", fontsize=8)
    fig.tight_layout()

    # share of the envelope's variance below / above the cut-off, and real vs noise
    slow, fast = f < SMOOTH_HZ, f >= SMOOTH_HZ
    table = pd.DataFrame({
        "envelope of": ["real component", "noise control"],
        f"variance below {SMOOTH_HZ} Hz (%)": [100 * p[slow].sum() / p.sum() for p in (p_real, p_noise)],
        f"variance above {SMOOTH_HZ} Hz (%)": [100 * p[fast].sum() / p.sum() for p in (p_real, p_noise)],
    }).round(1)
    print(table.to_string(index=False))
    print(f"real / noise power: below {SMOOTH_HZ} Hz x{p_real[slow].sum() / p_noise[slow].sum():.2f}, "
          f"above {SMOOTH_HZ} Hz x{p_real[fast].sum() / p_noise[fast].sum():.2f}")
else:
    print(f"{PLOT_AREA} not in SPATIAL_FILTER_CHOICE for {SUBJECT}: choose one of {list(CHOSEN)}")

## 4. Save

One file per area, with all events (for cutting trials later): `<sub>_<area>_<band>_<ICA>_envelope_raw.fif` (the ICA in the name, so envelopes from the session and the localizer ICAs can be compared). The choice goes to `<sub>_spatial_filter.csv` for the Methods.

In [ ]:
for area, x in envelopes.items():
    ica_name, comp, band = CHOSEN[area]
    x.save(OUT_DIR / f"{SUBJECT}_{area}_{band}_{ica_name}_envelope_raw.fif", overwrite=True)

if CHOSEN:
    choice = pd.DataFrame([{"area": area, "ica": ica_name, "component": comp, "band": band,
                            "transition_hz": cfg.SPATIAL_FILTER_TRANSITION, "envelope_sfreq": cfg.ENVELOPE_SFREQ}
                           for area, (ica_name, comp, band) in CHOSEN.items()])
    choice.to_csv(OUT_DIR / f"{SUBJECT}_spatial_filter.csv", index=False)
    choice

## Questions for Anna

1. Motor envelope: mu, beta, or both?
2. One motor component, or one per side (left and right hand)?
3. Choose components per participant by eye, or match them across participants with CORRMAP (a "visual" and a "motor" template), as for the artifacts? Which criteria should the choice by eye use?
4. SSD (spatio-spectral decomposition; Nikulin et al. 2011, Dähne et al. 2014) instead of ICA components, or as a check?
5. Compare with the simple alternative (average over ROI electrodes) as a check?